# 🚨 AOG 자재 수급 어시스턴트

채팅창에 상황을 말하면 AI가 조건을 읽어 즉시 1~6단계를 자동으로 훑고, 실제 연락/메일 발송이 필요한 지점에서만 승인을 구하는 Streamlit 앱입니다. 아래 3개 셀을 순서대로 실행하세요. 사용법·테스트 시나리오는 저장소의 `README.md`를 참고하세요.

## Cell 1 — 라이브러리 설치

**이전 버전과 달라진 점**: `--upgrade`와 강제 재시작(`os.kill`)을 없앴습니다. 두 가지 모두 이미 만족하는 Colab 기본 패키지(numpy/pandas 등)를 불필요하게 건드려 오히려 충돌을 만들 수 있는 요인이었습니다. 지금은 **어떤 사전 설치 패키지도 건드리지 않고, 아직 없는 패키지만 그대로 설치**합니다 — 설치 후 런타임을 재시작할 필요가 없습니다. 바로 Cell 2로 넘어가세요.

- `-q`(quiet)를 빼서 pip 출력이 전부 보이도록 했습니다 — 만약 설치가 실패하면, 이 셀의 출력(빨간 글씨 오류 메시지) 전체를 그대로 복사해서 알려주세요. 그래야 정확한 원인을 잡을 수 있습니다.
- `streamlit`, `plotly`는 필수, `transformers`는 선택(4·5단계 긴급 메일을 AI가 다듬는 기능에만 사용) 패키지라 설치 명령을 분리했습니다 — `transformers` 설치가 느리거나 실패해도 앱의 핵심 기능은 전혀 영향받지 않습니다.

> 그래도 위젯마다 `TypeError`가 난다면: Colab 메뉴에서 **런타임 → 세션 다시 시작**을 누른 뒤 Cell 1은 건너뛰고 Cell 2부터 다시 실행해보세요. (설치 자체는 이미 되어 있으므로 Cell 1을 다시 실행할 필요는 없습니다.)

In [ ]:
!pip install streamlit plotly
!pip install transformers


## Cell 2 — Streamlit 앱 코드 작성 (`app.py`)

In [ ]:
%%writefile app.py
# ============================================================================
# AOG(Aircraft on Ground) 자재 수급 어시스턴트 — Streamlit (v6, 묶음 데이터 모델)
# ----------------------------------------------------------------------------
# 데이터 모델(현실 반영):
#   - FAK 키트: 같은 기종이면 모든 기체가 '동일한 키트 하나(여러 자재 패키지)'를 기체와 한몸처럼
#     탑재한다. → fak_kits = [기종별 패키지 {aircraft_type, kit_name, contents:[자재...]}].
#     키트는 기체와 함께 이동하므로 AOG 발생 공항과 무관하게 현장에서 사용 가능.
#   - Allocation: 한 스테이션(공항) 창고 안에 여러 자재가 들어있다.
#     → station_warehouses = [공항별 창고 {airport, warehouse_name, contents:[자재...]}].
#     그 공항 창고에 있어야만 사용 가능. 없으면 Pooling으로 넘어간다.
#   - Pooling: 파트너가 재고 보유 + 커버리지 공항에 발생 공항 포함해야 지원 가능.
#   - 4단계: 발생 공항 Main Station 항공사 ∩ 우리 기종 운영사. 5단계: 기종 운영 전 항공사.
# 입력은 '기번(Registration)'. aircraft_registry가 기번→기종 매핑. 타사 문의는 기종 기준(영문 메일).
# ============================================================================

import copy
import json
import os
import re
import urllib.parse
from collections import Counter
from datetime import datetime, timedelta

import pandas as pd
import plotly.graph_objects as go
import streamlit as st

try:
    import transformers  # noqa: F401
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False

st.set_page_config(page_title="AOG 자재 수급 어시스턴트", page_icon="🚨", layout="wide")

DB_PATH = "aog_db.json"
SCHEMA_VERSION = "8"

# 공항 좌표 (위도, 경도, 한글명)
AIRPORT_COORDS = {
    "ICN": (37.4602, 126.4407, "인천"), "GMP": (37.5583, 126.7906, "김포"),
    "NRT": (35.7647, 140.3864, "나리타"), "HND": (35.5494, 139.7798, "하네다"),
    "CDG": (49.0097, 2.5479, "파리 CDG"), "JFK": (40.6413, -73.7781, "뉴욕 JFK"),
    "SIN": (1.3644, 103.9915, "싱가포르"), "HKG": (22.3080, 113.9185, "홍콩"),
    "FRA": (50.0379, 8.5622, "프랑크푸르트"), "LAX": (33.9416, -118.4085, "LA"),
    "BKK": (13.6900, 100.7501, "방콕"), "SYD": (-33.9399, 151.1753, "시드니"),
}

# ----------------------------------------------------------------------------
# 1-a. 기번 등록부 (기번 -> 기종)
# ----------------------------------------------------------------------------
_AIRCRAFT_REGISTRY = [
    {"registration": "HL7702", "aircraft_type": "A330-300"},
    {"registration": "HL7710", "aircraft_type": "A330-300"},
    {"registration": "HL7720", "aircraft_type": "A330-300"},
    {"registration": "HL8008", "aircraft_type": "B777-300ER"},
    {"registration": "HL8009", "aircraft_type": "B777-300ER"},
    {"registration": "HL8010", "aircraft_type": "B777-300ER"},
    {"registration": "HL8259", "aircraft_type": "B737-800"},
    {"registration": "HL8260", "aircraft_type": "B737-800"},
    {"registration": "HL8261", "aircraft_type": "B737-800"},
    {"registration": "HL8501", "aircraft_type": "A321neo"},
    {"registration": "HL8502", "aircraft_type": "A321neo"},
    {"registration": "HL8503", "aircraft_type": "A321neo"},
    {"registration": "HL8081", "aircraft_type": "B787-9"},
    {"registration": "HL8082", "aircraft_type": "B787-9"},
    {"registration": "HL8083", "aircraft_type": "B787-9"},
    {"registration": "HL8084", "aircraft_type": "B787-9"},
    {"registration": "HL8085", "aircraft_type": "B787-9"},
    {"registration": "HL8360", "aircraft_type": "A350-900"},
    {"registration": "HL8361", "aircraft_type": "A350-900"},
]

# ----------------------------------------------------------------------------
# 1-b. FAK 키트 표준 구성(기종별 패키지) — 같은 기종은 모두 동일 키트 하나를 탑재.
#      비상/소모성 품목(산소발생기, 연기감지기, 소화기, 구명조끼 등)이 한 묶음으로 들어있다.
# ----------------------------------------------------------------------------
_FAK_CONTENTS = {
    "A330-300": [
        ("OXY-GEN-A330-15", "Chemical Oxygen Generator", 2),
        ("SMK-DET-A330-21", "Cargo Compartment Smoke Detector", 3),
        ("LIFE-VEST-A330-40", "Passenger Life Vest", 8),
        ("FIRE-EXT-A330-33", "Portable Halon Fire Extinguisher", 4),
        ("O2-BOTTLE-A330-27", "Portable Oxygen Bottle", 5),
        ("ELT-A330-51", "Emergency Locator Transmitter", 1),
    ],
    "B777-300ER": [
        ("OXY-GEN-777-15", "Chemical Oxygen Generator", 2),
        ("SMK-DET-777-22", "Cargo Compartment Smoke Detector", 3),
        ("LIFE-VEST-777-40", "Passenger Life Vest", 10),
        ("FIRE-EXT-777-33", "Portable Halon Fire Extinguisher", 5),
        ("O2-BOTTLE-777-27", "Portable Oxygen Bottle", 6),
        ("ELT-777-51", "Emergency Locator Transmitter", 1),
    ],
    "B737-800": [
        ("OXY-GEN-737-15", "Chemical Oxygen Generator", 2),
        ("SMK-DET-737-04", "Cargo Compartment Smoke Detector", 3),
        ("LIFE-VEST-737-42", "Passenger Life Vest", 8),
        ("FIRE-EXT-737-33", "Portable Halon Fire Extinguisher", 3),
        ("O2-BOTTLE-737-08", "Portable Oxygen Bottle", 5),
        ("ELT-737-51", "Emergency Locator Transmitter", 1),
    ],
    "A321neo": [
        ("OXY-GEN-321N-15", "Chemical Oxygen Generator", 2),
        ("SMK-DET-321N-43", "Cargo Compartment Smoke Detector", 3),
        ("LIFE-VEST-321N-17", "Passenger Life Vest", 8),
        ("FIRE-EXT-321N-33", "Portable Halon Fire Extinguisher", 3),
        ("CAB-PRESS-VLV-321N-06", "Cabin Pressure Relief Valve", 2),
        ("ELT-321N-51", "Emergency Locator Transmitter", 1),
    ],
    "B787-9": [
        ("OXY-GEN-787-15", "Chemical Oxygen Generator", 2),
        ("SMK-DET-787-44", "Cargo Compartment Smoke Detector", 3),
        ("LIFE-VEST-787-40", "Passenger Life Vest", 10),
        ("FIRE-EXT-787-14", "Portable Halon Fire Extinguisher", 4),
        ("EVAC-SLIDE-787-02", "Emergency Evacuation Slide", 1),
        ("ELT-787-51", "Emergency Locator Transmitter", 1),
    ],
    "A350-900": [
        ("OXY-GEN-350-19", "Chemical Oxygen Generator", 2),
        ("SMOKE-DET-350-11", "Cargo Compartment Smoke Detector", 2),
        ("LIFE-VEST-350-45", "Passenger Life Vest", 10),
        ("FIRE-EXT-350-33", "Portable Halon Fire Extinguisher", 4),
        ("O2-BOTTLE-350-27", "Portable Oxygen Bottle", 6),
        ("ELT-350-51", "Emergency Locator Transmitter", 1),
    ],
}
_FAK_KITS = [
    {"aircraft_type": t, "kit_name": f"{t} 표준 FAK 패키지",
     "contents": [{"part_number": pn, "part_name": nm, "qty": q} for pn, nm, q in items]}
    for t, items in _FAK_CONTENTS.items()
]

# ----------------------------------------------------------------------------
# 1-c. 로터블(고가 교환품) 마스터(기종별 파트넘버->이름). Allocation·Pooling에서 이름 조회에 사용.
# ----------------------------------------------------------------------------
_ROTABLE_MASTER = {
    "A330-300": {
        "IDG-A330-001": "Integrated Drive Generator", "FUEL-NOZ-A330-07": "Engine Fuel Nozzle",
        "BRK-A330-CARBON-12": "Carbon Brake Assembly", "HYD-PUMP-A330-18": "Engine-Driven Hydraulic Pump",
        "WHL-MLG-A330-22": "Main Landing Gear Wheel", "STARTER-GEN-A330-24": "Starter Generator",
        "APU-A330-30": "Auxiliary Power Unit", "FMS-A330-40": "Flight Management Computer",
    },
    "B777-300ER": {
        "BRK-B777-CARBON-01": "Carbon Brake Assembly", "HYD-PUMP-777-23": "Engine-Driven Hydraulic Pump",
        "WHL-MLG-777-06": "Main Landing Gear Wheel", "IDG-777-11": "Integrated Drive Generator",
        "ENG-FAN-BLADE-777-31": "Engine Fan Blade", "APU-777-30": "Auxiliary Power Unit",
        "ADIRU-777-35": "Air Data Inertial Reference Unit", "FMS-777-40": "Flight Management Computer",
    },
    "B737-800": {
        "STARTER-GEN-737-03": "Starter Generator", "WHL-MLG-737-25": "Main Landing Gear Wheel",
        "HYD-PUMP-737-11": "Engine-Driven Hydraulic Pump", "BRK-737-CARBON-14": "Carbon Brake Assembly",
        "AVIONICS-FMS-737-32": "Flight Management Computer", "APU-737-30": "Auxiliary Power Unit",
    },
    "A321neo": {
        "FLAP-TRK-ROLLER-321N-08": "Flap Track Roller", "BRK-321N-CARBON-26": "Carbon Brake Assembly",
        "APU-321N-30": "Auxiliary Power Unit", "AVIONICS-FMS-321N-04": "Flight Management Computer",
        "WHL-NLG-321N-22": "Nose Landing Gear Wheel", "IDG-321N-11": "Integrated Drive Generator",
    },
    "B787-9": {
        "APU-787-01": "Auxiliary Power Unit", "IDG-787-27": "Integrated Drive Generator",
        "CARGO-DOOR-ACT-787-03": "Cargo Door Actuator", "BRK-787-CARBON-12": "Carbon Brake Assembly",
        "WHL-MLG-787-22": "Main Landing Gear Wheel", "FMS-787-40": "Flight Management Computer",
    },
    "A350-900": {
        "IDG-A350-002": "Integrated Drive Generator", "FUEL-PUMP-350-28": "Fuel Boost Pump",
        "BRK-350-CARBON-29": "Carbon Brake Assembly", "WHL-MLG-350-22": "Main Landing Gear Wheel",
        "APU-350-30": "Auxiliary Power Unit", "FMS-350-40": "Flight Management Computer",
    },
}
_ROTABLE_NAME = {pn: nm for parts in _ROTABLE_MASTER.values() for pn, nm in parts.items()}

# 스테이션별 보유 로터블 (airport, aircraft_type, part_number, qty) — 현실적 지리 분포(ICN 허브가 최다).
_STATION_STOCK = [
    ("ICN", "A330-300", "IDG-A330-001", 2), ("ICN", "A330-300", "FUEL-NOZ-A330-07", 4),
    ("ICN", "A330-300", "STARTER-GEN-A330-24", 1), ("ICN", "A330-300", "FMS-A330-40", 1),
    ("ICN", "B777-300ER", "BRK-B777-CARBON-01", 1), ("ICN", "B777-300ER", "HYD-PUMP-777-23", 2),
    ("ICN", "B777-300ER", "APU-777-30", 1), ("ICN", "B737-800", "STARTER-GEN-737-03", 2),
    ("ICN", "B737-800", "BRK-737-CARBON-14", 2), ("ICN", "A321neo", "FLAP-TRK-ROLLER-321N-08", 5),
    ("ICN", "A321neo", "BRK-321N-CARBON-26", 2), ("ICN", "B787-9", "APU-787-01", 1),
    ("ICN", "B787-9", "IDG-787-27", 1), ("ICN", "A350-900", "IDG-A350-002", 1),
    ("ICN", "A350-900", "FUEL-PUMP-350-28", 2),
    ("GMP", "B737-800", "WHL-MLG-737-25", 3), ("GMP", "A321neo", "WHL-NLG-321N-22", 2),
    ("NRT", "B787-9", "IDG-787-27", 1), ("NRT", "A330-300", "STARTER-GEN-A330-24", 1),
    ("NRT", "B777-300ER", "APU-777-30", 1),
    ("CDG", "A330-300", "FMS-A330-40", 1), ("CDG", "B787-9", "CARGO-DOOR-ACT-787-03", 1),
    ("FRA", "A330-300", "IDG-A330-001", 1), ("FRA", "B777-300ER", "ENG-FAN-BLADE-777-31", 1),
    ("LAX", "B777-300ER", "BRK-B777-CARBON-01", 1), ("LAX", "B787-9", "BRK-787-CARBON-12", 1),
    ("JFK", "B787-9", "WHL-MLG-787-22", 2), ("JFK", "B777-300ER", "WHL-MLG-777-06", 1),
    ("HKG", "B777-300ER", "HYD-PUMP-777-23", 1), ("HKG", "A350-900", "BRK-350-CARBON-29", 1),
    ("SIN", "A350-900", "FUEL-PUMP-350-28", 1), ("SIN", "A330-300", "FUEL-NOZ-A330-07", 1),
    ("BKK", "A330-300", "BRK-A330-CARBON-12", 1), ("SYD", "A350-900", "WHL-MLG-350-22", 1),
]
_WAREHOUSE_NAMES = {
    "ICN": "ICN 본사 통합 자재창고", "GMP": "GMP 지점 자재창고", "NRT": "NRT 해외지점 창고",
    "HND": "HND 해외지점 창고", "CDG": "CDG 해외지점 창고", "JFK": "JFK 해외지점 창고",
    "SIN": "SIN 해외지점 창고", "HKG": "HKG 해외지점 창고", "FRA": "FRA 해외지점 창고",
    "LAX": "LAX 해외지점 창고", "BKK": "BKK 해외지점 창고", "SYD": "SYD 해외지점 창고",
}


def _build_station_warehouses():
    by_ap = {}
    for ap, t, pn, qty in _STATION_STOCK:
        by_ap.setdefault(ap, []).append(
            {"aircraft_type": t, "part_number": pn, "part_name": _ROTABLE_NAME.get(pn, pn), "qty": qty})
    return [{"airport": ap, "warehouse_name": _WAREHOUSE_NAMES.get(ap, f"{ap} 자재창고"), "contents": items}
            for ap, items in by_ap.items()]


# ----------------------------------------------------------------------------
# 1-d. Pooling 파트너/재고
# ----------------------------------------------------------------------------
_POOLING_STOCK_RAW = [
    ("SIA Engineering (싱가포르)", "B737-800", "HYD-PUMP-737-11", 1, "SIN 창고"),
    ("SIA Engineering (싱가포르)", "A350-900", "BRK-350-CARBON-29", 1, "SIN 창고"),
    ("Lufthansa Technik (독일)", "A330-300", "IDG-A330-001", 1, "FRA 창고"),
    ("Lufthansa Technik (독일)", "A321neo", "APU-321N-30", 1, "FRA 창고"),
    ("HAECO (홍콩)", "B777-300ER", "WHL-MLG-777-06", 2, "HKG 창고"),
    ("AAR Corp (미국)", "A321neo", "AVIONICS-FMS-321N-04", 1, "JFK 창고"),
    ("AAR Corp (미국)", "B787-9", "WHL-MLG-787-22", 1, "JFK 창고"),
    ("ANA Base Maintenance (일본)", "B787-9", "CARGO-DOOR-ACT-787-03", 1, "NRT 창고"),
    ("MTU Maintenance (독일)", "B777-300ER", "ENG-FAN-BLADE-777-31", 1, "FRA 창고"),
    ("Delta TechOps (미국)", "B737-800", "AVIONICS-FMS-737-32", 1, "JFK 창고"),
    ("Delta TechOps (미국)", "A330-300", "FUEL-NOZ-A330-07", 1, "JFK 창고"),
]
_POOLING_STOCK = [
    {"partner": p, "aircraft_type": t, "part_number": pn, "part_name": _ROTABLE_NAME.get(pn, pn),
     "qty": q, "location": loc}
    for p, t, pn, q, loc in _POOLING_STOCK_RAW
]

# ----------------------------------------------------------------------------
# 1-e. DEFAULT_DB
# ----------------------------------------------------------------------------
DEFAULT_DB = {
    "_schema_version": SCHEMA_VERSION,
    "aircraft_registry": _AIRCRAFT_REGISTRY,
    "fak_kits": _FAK_KITS,                    # 기종별 키트 패키지(contents 묶음)
    "station_warehouses": _build_station_warehouses(),  # 공항별 창고(contents 묶음)
    "pooling_partners": [
        {"partner": "SIA Engineering (싱가포르)", "contact": "+65-6541-2000", "email": "poolingdesk@siae.com.sg", "coverage_airports": "SIN,HKG,BKK,NRT,ICN,SYD"},
        {"partner": "Lufthansa Technik (독일)", "contact": "+49-40-5070-5553", "email": "pooling@lht.dlh.de", "coverage_airports": "FRA,CDG"},
        {"partner": "ANA Base Maintenance (일본)", "contact": "+81-3-6735-1111", "email": "pooling@ana.co.jp", "coverage_airports": "NRT,HND,ICN"},
        {"partner": "HAECO (홍콩)", "contact": "+852-2767-6111", "email": "pooling@haeco.com", "coverage_airports": "HKG,SIN,BKK"},
        {"partner": "AAR Corp (미국)", "contact": "+1-630-227-2000", "email": "pooling@aarcorp.com", "coverage_airports": "JFK,LAX"},
        {"partner": "MTU Maintenance (독일)", "contact": "+49-511-7807-0", "email": "pooling@mtu.de", "coverage_airports": "FRA,CDG"},
        {"partner": "Delta TechOps (미국)", "contact": "+1-404-715-2600", "email": "pooling@delta.com", "coverage_airports": "JFK,LAX,ICN"},
    ],
    "pooling_stock": _POOLING_STOCK,
    "main_station_airlines": [
        {"airport": "ICN", "airline": "아시아나항공", "contact": "02-2669-8000", "email": "ops.icn@flyasiana.com"},
        {"airport": "GMP", "airline": "제주항공", "contact": "02-2015-1000", "email": "ops.gmp@jejuair.net"},
        {"airport": "NRT", "airline": "ANA", "contact": "+81-3-6735-1000", "email": "ops.nrt@ana.co.jp"},
        {"airport": "HND", "airline": "JAL", "contact": "+81-3-5460-3121", "email": "ops.hnd@jal.com"},
        {"airport": "CDG", "airline": "Air France", "contact": "+33-1-4356-7890", "email": "ops.cdg@airfrance.fr"},
        {"airport": "JFK", "airline": "Delta Air Lines", "contact": "+1-800-221-1212", "email": "ops.jfk@delta.com"},
        {"airport": "SIN", "airline": "Singapore Airlines", "contact": "+65-6223-8888", "email": "ops.sin@singaporeair.com"},
        {"airport": "HKG", "airline": "Cathay Pacific", "contact": "+852-2747-1888", "email": "ops.hkg@cathaypacific.com"},
        {"airport": "FRA", "airline": "Lufthansa", "contact": "+49-69-86799799", "email": "ops.fra@lufthansa.com"},
        {"airport": "LAX", "airline": "United Airlines", "contact": "+1-800-864-8331", "email": "ops.lax@united.com"},
        {"airport": "BKK", "airline": "Thai Airways", "contact": "+66-2-356-1111", "email": "ops.bkk@thaiairways.com"},
        {"airport": "SYD", "airline": "Qantas", "contact": "+61-2-9691-3636", "email": "ops.syd@qantas.com.au"},
    ],
    "fleet_operators": [
        {"aircraft_type": "A330-300", "airline": "아시아나항공", "contact": "02-2669-8000", "email": "fleet.a330@flyasiana.com"},
        {"aircraft_type": "A330-300", "airline": "Cathay Pacific", "contact": "+852-2747-1888", "email": "fleet.a330@cathaypacific.com"},
        {"aircraft_type": "A330-300", "airline": "China Eastern", "contact": "+86-21-95530", "email": "fleet.a330@ceair.com"},
        {"aircraft_type": "A330-300", "airline": "Thai Airways", "contact": "+66-2-356-1111", "email": "fleet.a330@thaiairways.com"},
        {"aircraft_type": "B777-300ER", "airline": "아시아나항공", "contact": "02-2669-8000", "email": "fleet.b777@flyasiana.com"},
        {"aircraft_type": "B777-300ER", "airline": "Emirates", "contact": "+971-600-555555", "email": "fleet.b777@emirates.com"},
        {"aircraft_type": "B777-300ER", "airline": "ANA", "contact": "+81-3-6735-1000", "email": "fleet.b777@ana.co.jp"},
        {"aircraft_type": "B777-300ER", "airline": "Cathay Pacific", "contact": "+852-2747-1888", "email": "fleet.b777@cathaypacific.com"},
        {"aircraft_type": "B737-800", "airline": "제주항공", "contact": "02-2015-1000", "email": "fleet.b737@jejuair.net"},
        {"aircraft_type": "B737-800", "airline": "티웨이항공", "contact": "1688-8686", "email": "fleet.b737@twayair.com"},
        {"aircraft_type": "B737-800", "airline": "Ryanair", "contact": "+353-1-945-1212", "email": "fleet.b737@ryanair.com"},
        {"aircraft_type": "B737-800", "airline": "Qantas", "contact": "+61-2-9691-3636", "email": "fleet.b737@qantas.com.au"},
        {"aircraft_type": "A321neo", "airline": "에어부산", "contact": "1666-3060", "email": "fleet.a321n@airbusan.com"},
        {"aircraft_type": "A321neo", "airline": "JAL", "contact": "+81-3-5460-3121", "email": "fleet.a321n@jal.com"},
        {"aircraft_type": "A321neo", "airline": "Wizz Air", "contact": "+36-1-777-9300", "email": "fleet.a321n@wizzair.com"},
        {"aircraft_type": "B787-9", "airline": "ANA", "contact": "+81-3-6735-1000", "email": "fleet.b787@ana.co.jp"},
        {"aircraft_type": "B787-9", "airline": "United Airlines", "contact": "+1-800-864-8331", "email": "fleet.b787@united.com"},
        {"aircraft_type": "B787-9", "airline": "JAL", "contact": "+81-3-5460-3121", "email": "fleet.b787@jal.com"},
        {"aircraft_type": "B787-9", "airline": "Air France", "contact": "+33-1-4356-7890", "email": "fleet.b787@airfrance.fr"},
        {"aircraft_type": "A350-900", "airline": "아시아나항공", "contact": "02-2669-8000", "email": "fleet.a350@flyasiana.com"},
        {"aircraft_type": "A350-900", "airline": "Cathay Pacific", "contact": "+852-2747-1888", "email": "fleet.a350@cathaypacific.com"},
        {"aircraft_type": "A350-900", "airline": "Qatar Airways", "contact": "+974-4023-0000", "email": "fleet.a350@qatarairways.com.qa"},
        {"aircraft_type": "A350-900", "airline": "Singapore Airlines", "contact": "+65-6223-8888", "email": "fleet.a350@singaporeair.com"},
        {"aircraft_type": "A350-900", "airline": "Thai Airways", "contact": "+66-2-356-1111", "email": "fleet.a350@thaiairways.com"},
    ],
    "allocation_dept_contacts": [
        {"airport": "ICN", "department": "자재관리팀 Allocation 파트", "contact": "02-XXXX-1000", "email": "allocation.icn@airline.example"},
        {"airport": "GMP", "department": "자재관리팀 Allocation 파트(김포)", "contact": "02-XXXX-1005", "email": "allocation.gmp@airline.example"},
        {"airport": "NRT", "department": "해외지점 Allocation 파트(나리타)", "contact": "+81-3-XXXX-1010", "email": "allocation.nrt@airline.example"},
        {"airport": "CDG", "department": "해외지점 Allocation 파트(파리)", "contact": "+33-1-XXXX-1015", "email": "allocation.cdg@airline.example"},
        {"airport": "HKG", "department": "해외지점 Allocation 파트(홍콩)", "contact": "+852-XXXX-1020", "email": "allocation.hkg@airline.example"},
        {"airport": "SIN", "department": "해외지점 Allocation 파트(싱가포르)", "contact": "+65-XXXX-1025", "email": "allocation.sin@airline.example"},
        {"airport": "FRA", "department": "해외지점 Allocation 파트(프랑크푸르트)", "contact": "+49-69-XXXX-1030", "email": "allocation.fra@airline.example"},
        {"airport": "LAX", "department": "해외지점 Allocation 파트(LA)", "contact": "+1-310-XXXX-1035", "email": "allocation.lax@airline.example"},
        {"airport": "JFK", "department": "해외지점 Allocation 파트(뉴욕)", "contact": "+1-718-XXXX-1040", "email": "allocation.jfk@airline.example"},
        {"airport": "NRT", "department": "해외지점 Allocation 파트(나리타)", "contact": "+81-3-XXXX-1010", "email": "allocation.nrt@airline.example"},
    ],
    "customs_team": {"name": "통관팀", "contact": "02-XXXX-2000", "email": "customs@airline.example"},

    # (8) 과거 수배(빌린) 이력: 같은 자재를 과거에 어디서·어느 단계·어떤 방식(대여/구매/Hand-carry/
    #     자체재고)으로 확보했고 성공했는지. 담당자가 "이 자재는 지난번 어디서 빌려서 됐지?"를 즉시 참고.
    "sourcing_history": [
        {"date": "2026-06-25", "registration": "HL7702", "aircraft_type": "A330-300", "part_number": "OXY-GEN-A330-15", "airport": "CDG", "resolved_step": "1·FAK", "source": "기체 탑재 FAK 키트", "method": "자체재고(FAK)", "result": "성공", "lead_time_hours": 1},
        {"date": "2026-05-12", "registration": "HL7710", "aircraft_type": "A330-300", "part_number": "IDG-A330-001", "airport": "FRA", "resolved_step": "3·Pooling", "source": "Lufthansa Technik (FRA)", "method": "대여(Loan)", "result": "성공", "lead_time_hours": 6},
        {"date": "2026-02-20", "registration": "HL7702", "aircraft_type": "A330-300", "part_number": "IDG-A330-001", "airport": "ICN", "resolved_step": "2·Allocation", "source": "ICN 본사 통합 자재창고", "method": "자체재고(Allocation)", "result": "성공", "lead_time_hours": 2},
        {"date": "2026-06-01", "registration": "HL7720", "aircraft_type": "A330-300", "part_number": "FUEL-NOZ-A330-07", "airport": "ICN", "resolved_step": "2·Allocation", "source": "ICN 본사 통합 자재창고", "method": "자체재고(Allocation)", "result": "성공", "lead_time_hours": 3},
        {"date": "2026-03-15", "registration": "HL8259", "aircraft_type": "B737-800", "part_number": "HYD-PUMP-737-11", "airport": "SIN", "resolved_step": "3·Pooling", "source": "SIA Engineering (SIN)", "method": "대여(Loan)", "result": "성공", "lead_time_hours": 8},
        {"date": "2026-01-09", "registration": "HL8260", "aircraft_type": "B737-800", "part_number": "HYD-PUMP-737-11", "airport": "JFK", "resolved_step": "6·Hand-carry", "source": "본사 ICN→JFK KE081", "method": "Hand-carry", "result": "성공", "lead_time_hours": 26},
        {"date": "2026-04-22", "registration": "HL8008", "aircraft_type": "B777-300ER", "part_number": "BRK-B777-CARBON-01", "airport": "ICN", "resolved_step": "2·Allocation", "source": "ICN 본사 통합 자재창고", "method": "자체재고(Allocation)", "result": "성공", "lead_time_hours": 2},
        {"date": "2026-05-30", "registration": "HL8009", "aircraft_type": "B777-300ER", "part_number": "WHL-MLG-777-06", "airport": "HKG", "resolved_step": "3·Pooling", "source": "HAECO (HKG)", "method": "대여(Loan)", "result": "성공", "lead_time_hours": 5},
        {"date": "2025-12-11", "registration": "HL8081", "aircraft_type": "B787-9", "part_number": "CARGO-DOOR-ACT-787-03", "airport": "NRT", "resolved_step": "3·Pooling", "source": "ANA Base Maintenance (NRT)", "method": "대여(Loan)", "result": "성공", "lead_time_hours": 7},
        {"date": "2026-06-18", "registration": "HL8082", "aircraft_type": "B787-9", "part_number": "APU-787-01", "airport": "LAX", "resolved_step": "5·타 항공사", "source": "United Airlines (LAX)", "method": "대여(Loan)", "result": "실패", "lead_time_hours": 6},
        {"date": "2026-06-19", "registration": "HL8082", "aircraft_type": "B787-9", "part_number": "APU-787-01", "airport": "LAX", "resolved_step": "6·Hand-carry", "source": "본사 ICN→LAX KE011", "method": "Hand-carry", "result": "성공", "lead_time_hours": 30},
        {"date": "2026-07-02", "registration": "HL8084", "aircraft_type": "B787-9", "part_number": "CARGO-DOOR-ACT-787-03", "airport": "NRT", "resolved_step": "3·Pooling", "source": "ANA Base Maintenance (NRT)", "method": "대여(Loan)", "result": "성공", "lead_time_hours": 5},
        {"date": "2026-06-30", "registration": "HL8501", "aircraft_type": "A321neo", "part_number": "AVIONICS-FMS-321N-04", "airport": "JFK", "resolved_step": "3·Pooling", "source": "AAR Corp (JFK)", "method": "대여(Loan)", "result": "성공", "lead_time_hours": 9},
    ],

    # (9) 과거 결함 조치 이력: 같은 자재/결함을 과거에 어떻게 처리했는지.
    #     resolution(부품 교환/디퍼(MEL)/리셋·재시동/SW 리로드), parts_scope(단일 부품 vs 패키지 어셈블리),
    #     tools_required(전용 공구)까지 기록 → "파츠 하나만 바꾸면 되는지, 패키지로 가야 하는지, 공구는 뭐가
    #     필요한지"를 진행 중에 바로 참고. (현재는 더미, 추후 정비 sheet 연동 가정)
    "defect_history": [
        {"date": "2026-06-25", "registration": "HL7702", "aircraft_type": "A330-300", "part_number": "OXY-GEN-A330-15", "defect": "Oxygen generator past expiry", "resolution": "부품 교환", "parts_scope": "단일 부품", "tools_required": "표준 공구", "result": "정상 복구", "downtime_hours": 1},
        {"date": "2026-05-12", "registration": "HL7710", "aircraft_type": "A330-300", "part_number": "IDG-A330-001", "defect": "IDG low oil pressure warning", "resolution": "부품 교환", "parts_scope": "패키지(어셈블리)", "tools_required": "IDG 인출 지그 JIG-IDG-01, 토크렌치", "result": "정상 복구", "downtime_hours": 6},
        {"date": "2026-02-20", "registration": "HL7702", "aircraft_type": "A330-300", "part_number": "IDG-A330-001", "defect": "IDG disconnect fault (intermittent)", "resolution": "리셋/재시동", "parts_scope": "교환 없음(리셋)", "tools_required": "BITE 테스터", "result": "정상 복구", "downtime_hours": 1},
        {"date": "2026-06-01", "registration": "HL7720", "aircraft_type": "A330-300", "part_number": "FUEL-NOZ-A330-07", "defect": "Fuel nozzle coking / uneven spray", "resolution": "부품 교환", "parts_scope": "단일 부품", "tools_required": "노즐 풀러 PULLER-07", "result": "정상 복구", "downtime_hours": 3},
        {"date": "2026-03-15", "registration": "HL8259", "aircraft_type": "B737-800", "part_number": "HYD-PUMP-737-11", "defect": "Hydraulic pump low pressure", "resolution": "부품 교환", "parts_scope": "패키지(어셈블리)", "tools_required": "유압 실링 키트, 토크렌치", "result": "정상 복구", "downtime_hours": 8},
        {"date": "2026-05-05", "registration": "HL8260", "aircraft_type": "B737-800", "part_number": "SMK-DET-737-04", "defect": "Cargo smoke detector false warning", "resolution": "리셋/재시동", "parts_scope": "교환 없음(리셋)", "tools_required": "표준 공구", "result": "정상 복구", "downtime_hours": 1},
        {"date": "2026-04-22", "registration": "HL8008", "aircraft_type": "B777-300ER", "part_number": "BRK-B777-CARBON-01", "defect": "Brake wear beyond limit", "resolution": "부품 교환", "parts_scope": "단일 부품", "tools_required": "브레이크 잭 JACK-BR-02", "result": "정상 복구", "downtime_hours": 2},
        {"date": "2026-06-18", "registration": "HL8082", "aircraft_type": "B787-9", "part_number": "APU-787-01", "defect": "APU start fault", "resolution": "디퍼(MEL) 후 부품 교환", "parts_scope": "패키지(어셈블리)", "tools_required": "APU 호이스트 HOIST-APU-01", "result": "재발 → 부품 교환", "downtime_hours": 30},
        {"date": "2026-07-02", "registration": "HL8084", "aircraft_type": "B787-9", "part_number": "CARGO-DOOR-ACT-787-03", "defect": "Cargo door actuator fail to latch", "resolution": "리셋/재시동", "parts_scope": "교환 없음(리셋)", "tools_required": "BITE 테스터", "result": "정상 복구", "downtime_hours": 1},
        {"date": "2025-12-11", "registration": "HL8081", "aircraft_type": "B787-9", "part_number": "CARGO-DOOR-ACT-787-03", "defect": "Cargo door actuator fail to latch", "resolution": "SW 리로드 후 부품 교환", "parts_scope": "패키지(어셈블리)", "tools_required": "액추에이터 리깅 툴 RIG-DR-03", "result": "정상 복구", "downtime_hours": 7},
        {"date": "2026-05-30", "registration": "HL8009", "aircraft_type": "B777-300ER", "part_number": "WHL-MLG-777-06", "defect": "Tire/wheel damage on landing", "resolution": "부품 교환", "parts_scope": "단일 부품", "tools_required": "휠 밸런서, 토크렌치", "result": "정상 복구", "downtime_hours": 5},
    ],
}

STEP_NAMES = {
    1: "1단계 · FAK 키트 확인",
    2: "2단계 · Allocation(스테이션 창고) 확인",
    3: "3단계 · Pooling 파트너사 확인",
    4: "4단계 · 발생 공항 Main Station 타 항공사 문의",
    5: "5단계 · 동일 기종 운영 타 항공사 문의",
    6: "6단계 · 본사 Hand-carry 이동",
}
TOTAL_STEPS = 6

DEFAULT_EMAIL_SUBJECT = "[AOG - URGENT] Spare part support request - {aircraft_type} / {part_number} / {airport}"
DEFAULT_EMAIL_BODY_TEMPLATE = (
    "Dear Operations / Material Support Team,\n\n"
    "We are currently experiencing an AOG (Aircraft on Ground) situation on our {aircraft_type} "
    "aircraft at {airport}. We urgently require the following part and would greatly appreciate "
    "your support:\n\n"
    "  - Part Number: {part_number}\n"
    "  - Aircraft Type: {aircraft_type}\n"
    "  - Station: {airport}\n\n"
    "If you have this part available, please advise on loan availability and the minimum lead time "
    "at your earliest convenience.\n\n"
    "Thank you for your urgent assistance.\n\n"
    "Best regards,\n"
    "Material Control Department"
)

AIRCRAFT_TYPES = sorted({r["aircraft_type"] for r in DEFAULT_DB["aircraft_registry"]})
REGISTRATION_CATALOG = [r["registration"] for r in DEFAULT_DB["aircraft_registry"]]
AIRPORTS = list(AIRPORT_COORDS.keys())
PART_NUMBER_CATALOG = sorted(
    {it["part_number"] for k in DEFAULT_DB["fak_kits"] for it in k["contents"]}
    | {it["part_number"] for w in DEFAULT_DB["station_warehouses"] for it in w["contents"]}
    | {r["part_number"] for r in DEFAULT_DB["pooling_stock"]}
)


# ----------------------------------------------------------------------------
# 2. 영속화 (스키마 버전 불일치 시 백업 후 재생성)
# ----------------------------------------------------------------------------

def _migrate_db(db):
    for r in db.get("pooling_partners", []):
        r.setdefault("coverage_airports", "")
    return db


def load_db():
    if os.path.exists(DB_PATH):
        try:
            with open(DB_PATH, "r", encoding="utf-8") as f:
                db = json.load(f)
            if db.get("_schema_version") == SCHEMA_VERSION:
                return _migrate_db(db)
            os.replace(DB_PATH, DB_PATH + ".bak")
        except Exception:
            pass
    db = copy.deepcopy(DEFAULT_DB)
    save_db(db)
    return db


def save_db(db):
    db["_schema_version"] = SCHEMA_VERSION
    with open(DB_PATH, "w", encoding="utf-8") as f:
        json.dump(db, f, ensure_ascii=False, indent=2)


if "db" not in st.session_state:
    st.session_state.db = load_db()
if "case" not in st.session_state:
    st.session_state.case = None
if "chat_log" not in st.session_state:
    st.session_state.chat_log = []


# ----------------------------------------------------------------------------
# 3. 데이터 조회 헬퍼
# ----------------------------------------------------------------------------

def norm(s):
    return (s or "").strip().upper()


def tel_href(contact):
    return re.sub(r"[^0-9+]", "", contact or "")


def find_one(records, **filters):
    for r in records:
        if all(norm(r.get(k)) == norm(v) for k, v in filters.items()):
            return r
    return None


def find_all(records, **filters):
    return [r for r in records if all(norm(r.get(k)) == norm(v) for k, v in filters.items())]


def _content_find(bundle, **filters):
    """묶음(패키지/창고)의 contents에서 조건에 맞는 자재 항목을 찾는다."""
    if not bundle:
        return None
    for it in bundle.get("contents", []):
        if all(norm(it.get(k)) == norm(v) for k, v in filters.items()):
            return it
    return None


def resolve_aircraft_type(db, registration):
    row = find_one(db["aircraft_registry"], registration=registration)
    if row:
        return row["aircraft_type"]
    for r in db["aircraft_registry"]:
        if norm(r["aircraft_type"]) == norm(registration):
            return r["aircraft_type"]
    return None


def airlines_operating_type(db, aircraft_type):
    return {norm(r["airline"]): r for r in db["fleet_operators"] if norm(r["aircraft_type"]) == norm(aircraft_type)}


def allocation_find_anywhere(db, aircraft_type, part_number):
    for wh in db["station_warehouses"]:
        it = _content_find(wh, aircraft_type=aircraft_type, part_number=part_number)
        if it:
            return wh, it
    return None, None


# ----------------------------------------------------------------------------
# 4. 단계별 조회 로직 — 전부 데이터(묶음) 기반.
# ----------------------------------------------------------------------------

def evaluate_step(step, db, registration, aircraft_type, part_number, airport):
    if step == 1:
        # 같은 기종은 모두 동일한 FAK 키트 하나를 기체와 함께 탑재 → 기종 패키지의 contents에서 조회.
        kit = find_one(db["fak_kits"], aircraft_type=aircraft_type)
        item = _content_find(kit, part_number=part_number)
        if item:
            detail = {**item, "kit_name": kit.get("kit_name", "")}
            return True, (f"✅ **FAK 키트 보유** — {registration}({aircraft_type}) 탑재 {kit.get('kit_name', '표준 키트')}에 포함.\n"
                           f"- 자재: {item.get('part_name', part_number)} · 키트 내 수량: {item['qty']}개 · 현장: {airport}\n\n"
                           f"동일 기종은 모두 같은 키트를 기체와 함께 탑재하므로 {airport} 현장에서 즉시 사용 가능."), detail
        return False, f"❌ 이 자재는 {aircraft_type} 표준 FAK 키트 구성품이 아닙니다.", None

    if step == 2:
        # 발생 공항 스테이션 창고 안에 해당 (기종·자재)가 있어야 사용 가능. 없으면 Pooling으로.
        wh = find_one(db["station_warehouses"], airport=airport)
        item = _content_find(wh, aircraft_type=aircraft_type, part_number=part_number)
        if item:
            detail = {**item, "warehouse_name": wh.get("warehouse_name", ""), "airport": airport}
            return True, (f"✅ **Allocation 재고 확인** — {airport} {wh.get('warehouse_name', '창고')}\n"
                           f"- 자재: {item.get('part_name', part_number)} · 수량: {item['qty']}개\n\n"
                           f"{airport} 스테이션 창고 재고로 조치 가능."), detail
        owh, oitem = allocation_find_anywhere(db, aircraft_type, part_number)
        if oitem:
            return False, (f"❌ {airport} 스테이션 창고에는 없습니다. "
                            f"(참고: {owh['airport']} {owh.get('warehouse_name', '창고')}에 {oitem['qty']}개 있으나 "
                            f"{airport}에서 즉시 쓸 수 없어 Pooling으로 넘어갑니다.)"), None
        return False, f"❌ 어느 스테이션 창고에도 {aircraft_type} {part_number} 재고가 없습니다.", None

    if step == 3:
        candidates = find_all(db["pooling_stock"], aircraft_type=aircraft_type, part_number=part_number)
        for hit in candidates:
            pinfo = find_one(db["pooling_partners"], partner=hit["partner"]) or {}
            coverage = [c.strip() for c in pinfo.get("coverage_airports", "").split(",") if c.strip()]
            if not coverage or norm(airport) in [norm(c) for c in coverage]:
                detail = {**hit, "contact": pinfo.get("contact", ""), "email": pinfo.get("email", ""),
                          "coverage_airports": pinfo.get("coverage_airports", "")}
                return True, (f"✅ **Pooling 지원 가능 — {hit['partner']}**\n"
                               f"- 자재: {hit.get('part_name', part_number)} · 보유: {hit['location']} · 수량: {hit['qty']}개\n"
                               f"- 커버리지: {pinfo.get('coverage_airports', '전 공항')} · 연락처: {detail['contact']}\n\n"
                               f"{airport}은(는) 이 파트너의 서비스 지역이라 대여 요청이 가능합니다."), detail
        if candidates:
            hit = candidates[0]
            pinfo = find_one(db["pooling_partners"], partner=hit["partner"]) or {}
            return False, (f"❌ {airport}은(는) 이 자재를 보유한 Pooling 파트너의 서비스 지역이 아닙니다. "
                            f"(참고: {hit['partner']}가 {hit['location']}에 {hit['qty']}개 보유하나, "
                            f"커버리지 {pinfo.get('coverage_airports', '미등록')}에 {airport}이(가) 없습니다.)"), None
        return False, "❌ 해당 자재를 보유한 Pooling 파트너사가 없습니다.", None

    if step == 4:
        operators = airlines_operating_type(db, aircraft_type)
        station_airlines = find_all(db["main_station_airlines"], airport=airport)
        queried = [a for a in station_airlines if norm(a["airline"]) in operators]
        detail = {"queried_airlines": queried}
        if queried:
            names = ", ".join(a["airline"] for a in queried)
            return False, (f"ℹ️ {airport} 기반 항공사 중 {aircraft_type} 운영사({names})에 대여 문의가 필요합니다. "
                            f"타사 실재고는 사전에 알 수 없어 직접 문의로 확인합니다."), detail
        station_names = ", ".join(a["airline"] for a in station_airlines) or "등록된 항공사 없음"
        return False, (f"❌ {airport} Main Station 항공사({station_names}) 중 {aircraft_type} 운영사가 없습니다. "
                        f"5단계로 넘어갑니다."), detail

    if step == 5:
        operators = list(airlines_operating_type(db, aircraft_type).values())
        detail = {"queried_airlines": operators}
        names = ", ".join(o["airline"] for o in operators) or "등록된 운영사 없음"
        return False, (f"ℹ️ {aircraft_type} 운영 항공사({names}) 전체에 대여 문의가 필요합니다. "
                        f"타사 실재고는 기밀이라 개별 문의로 확인합니다."), detail

    if step == 6:
        flights = fetch_flight_schedule(airport)[:3]
        detail = {"flights": flights, "customs": db["customs_team"]}
        return True, ("🧳 **본사 Hand-carry 이동 확정**\n"
                       "이전 단계에서 확보하지 못해 최종 수단으로 조치합니다. 통관팀과 가장 빠른 편명을 확인하세요."), detail

    raise ValueError(f"알 수 없는 단계: {step}")


# ----------------------------------------------------------------------------
# 5. 실시간 항공편 / 공항 데이터 공급자 (더미 — 실 API 연동 지점 표시)
# ----------------------------------------------------------------------------

FLIGHT_API_CONFIG = {"enabled": False, "base_url": "https://api.example-airline.com/v1/flights", "api_key": ""}
PARKING_API_CONFIG = {"enabled": False, "base_url": "https://api.example-airline.com/v1/stands", "api_key": ""}

_ROUTES = [
    ("ICN", "LAX", 11, 3, 11.2), ("ICN", "JFK", 81, 2, 14.0), ("ICN", "CDG", 901, 2, 12.5),
    ("ICN", "FRA", 905, 2, 11.7), ("ICN", "SIN", 643, 2, 6.5), ("ICN", "HKG", 603, 3, 3.7),
    ("ICN", "NRT", 705, 3, 2.5), ("ICN", "BKK", 651, 2, 5.9), ("ICN", "SYD", 121, 2, 10.5),
    ("ICN", "HND", 719, 2, 2.5), ("GMP", "HND", 2711, 2, 2.6),
]
_OFFSET_CYCLE = [-2.0, 6.0, -0.5, 8.0, 3.0, -4.0, 10.0, 1.5, -1.0, 5.0, 12.0, -3.0,
                 4.0, 9.0, -5.0, 7.0, 2.0, 11.0, -2.5, 6.5, 0.5, -6.0, 13.0, 3.5]
_PARKING_STAND_CAPACITY = {
    "ICN": 46, "GMP": 20, "NRT": 30, "HND": 28, "CDG": 60, "JFK": 55,
    "SIN": 35, "HKG": 40, "FRA": 58, "LAX": 50, "BKK": 32, "SYD": 26,
}


def _generate_dummy_live_flights():
    now = datetime.now()
    flights, idx = [], 0
    for origin, dest, base, freq, dur in _ROUTES:
        for k in range(freq):
            for (o, d, no) in [(origin, dest, base + 2 * k), (dest, origin, base + 2 * k + 1)]:
                off = _OFFSET_CYCLE[idx % len(_OFFSET_CYCLE)]
                idx += 1
                dep = now + timedelta(hours=off)
                arr = dep + timedelta(hours=dur)
                if now < dep:
                    status, prog = "예정", 0.0
                elif now >= arr:
                    status, prog = "도착", 1.0
                else:
                    status = "비행 중"
                    prog = (now - dep).total_seconds() / (arr - dep).total_seconds()
                flights.append({
                    "flight_no": f"KE{no:04d}", "airline": "대한항공", "origin": o, "destination_airport": d,
                    "dep_time": dep, "arr_time": arr, "status": status, "progress": round(prog, 3),
                    "hours_from_now": round(max((arr - now).total_seconds() / 3600, 0.0), 1),
                })
    return flights


@st.cache_data(ttl=60, show_spinner=False)
def _fetch_raw_flight_feed():
    if FLIGHT_API_CONFIG["enabled"]:
        raise NotImplementedError("FLIGHT_API_CONFIG.enabled=True면 실제 연동 로직을 구현하세요.")
    return _generate_dummy_live_flights()


def fetch_live_flights():
    return _fetch_raw_flight_feed()


def fetch_flight_schedule(destination_airport=None):
    flights = [f for f in fetch_live_flights() if f["status"] == "예정"]
    if destination_airport:
        flights = [f for f in flights if norm(f["destination_airport"]) == norm(destination_airport)]
    return sorted(flights, key=lambda f: f["hours_from_now"])


@st.cache_data(ttl=60, show_spinner=False)
def fetch_airport_parking_status():
    if PARKING_API_CONFIG["enabled"]:
        raise NotImplementedError("PARKING_API_CONFIG.enabled=True면 실제 연동 로직을 구현하세요.")
    import random
    rng = random.Random(datetime.now().strftime("%Y%m%d%H%M"))
    return {ap: {"total_stands": total, "occupied": rng.randint(int(total * 0.4), int(total * 0.95))}
            for ap, total in _PARKING_STAND_CAPACITY.items()}


# ----------------------------------------------------------------------------
# 6. AI 브리핑(즉시·규칙 기반) + 로컬 LLM(선택: 4·5단계 영문 메일 다듬기)
# ----------------------------------------------------------------------------

def build_rule_based_briefing(case):
    lines = [
        "### 📌 케이스 개요",
        f"- 기번: **{case['registration']}** → 기종: **{case['aircraft_type']}**",
        f"- 자재: **{case['part_number']}** · 발생 공항: **{case['airport']}**",
        f"- 현재 단계: **{case['step']}/{TOTAL_STEPS} — {STEP_NAMES[case['step']]}**",
        "",
        "### 🔍 지금까지 조회 결과",
    ]
    for rec in case["records"]:
        lines.append(f"- {'✅' if rec['found'] else '❌'} {STEP_NAMES[rec['step']]}")

    lines.append("")
    lines.append("### 💡 추천 행동")
    last = case["records"][-1]
    step = case["step"]

    if case["finished"]:
        if last["found"]:
            action = {
                1: f"현장 정비사에게 FAK 키트에 {case['part_number']}가 있음을 재확인·전달하세요.",
                2: f"{case['airport']} Allocation 부서에 연락해 스테이션 창고 자재 반출을 요청하세요.",
                3: "Pooling 파트너사에 연락해 대여 절차를 진행하세요.",
                6: "통관팀에 연락하고 가장 빠른 편명으로 Hand-carry를 확정하세요.",
            }.get(last["step"], "확보한 경로로 조치를 마무리하세요.")
            lines.append(f"- 🏁 **{STEP_NAMES[last['step']]}**에서 확보 완료. {action}")
        else:
            lines.append(f"- 🏁 **{STEP_NAMES[last['step']]}**에서 담당자가 종료했습니다(외부 확보 또는 대체 조달).")
    elif last["found"]:
        lines.append(f"- 🎯 **{STEP_NAMES[step]}**에서 확보 가능. 아래에서 승인하면 조치를 진행합니다.")
    elif step in (4, 5):
        n = len(last["detail"].get("queried_airlines", []))
        lines.append(f"- ✉️ 데이터에서 추린 문의 대상 **{n}개 항공사**에 긴급 대여 메일(영문·기종 기준)을 보내세요.")
        lines.append("- 🛡️ 6단계는 통관에 시간이 걸리므로, 지금 통관팀에 여유 자재 불출을 병행 요청하는 것을 권장합니다.")
    elif step >= TOTAL_STEPS:
        lines.append("- ▶ 마지막 조달 경로입니다. Hand-carry를 확정하세요.")
    else:
        lines.append(f"- ▶ 승인하면 **{STEP_NAMES[step + 1]}**로 진행합니다.")
    return "\n".join(lines)


@st.cache_resource(show_spinner=False)
def _load_local_llm():
    from transformers import pipeline
    return pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")


def generate_llm_email_body(case):
    try:
        pipe = _load_local_llm()
        messages = [
            {"role": "system", "content": (
                "You are a material control officer at an airline. Write only the body of a polite, "
                "concise English business email requesting AOG spare-part support from another airline. "
                "Keep it to 3-4 sentences, output the body text only (no subject line).")},
            {"role": "user", "content": (
                f"Aircraft type {case['aircraft_type']}, part number {case['part_number']}, station {case['airport']}. "
                f"We are short of this part; ask whether they have it available for loan.")},
        ]
        out = pipe(messages, max_new_tokens=120, do_sample=False)
        return (out[0]["generated_text"][-1]["content"] or "").strip() or None
    except Exception:
        return None


# ----------------------------------------------------------------------------
# 7. 케이스(프로세스) 상태 관리
# ----------------------------------------------------------------------------

def try_start_case(registration, part_number, airport, mechanic_contact):
    aircraft_type = resolve_aircraft_type(st.session_state.db, registration)
    if aircraft_type is None:
        return False, (f"기번 **{registration}**은(는) 등록부에 없습니다. "
                       f"'⚙️ 데이터 관리 → 🛩️ 기번 등록부'에 추가한 뒤 다시 시도하세요.")
    case = {
        "registration": registration.strip(), "aircraft_type": aircraft_type,
        "part_number": part_number.strip(), "airport": airport.strip().upper(),
        "mechanic_contact": mechanic_contact.strip(),
        "step": 1, "finished": False, "records": [], "action_log": [],
        "started_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    }
    found, message, detail = evaluate_step(1, st.session_state.db, case["registration"], aircraft_type, case["part_number"], case["airport"])
    case["records"].append({"step": 1, "found": found, "message": message, "detail": detail})
    st.session_state.case = case
    _auto_sweep_silent_steps()
    return True, None


def proceed_step():
    case = st.session_state.case
    if case["finished"] or case["step"] >= TOTAL_STEPS:
        return
    nxt = case["step"] + 1
    found, message, detail = evaluate_step(nxt, st.session_state.db, case["registration"], case["aircraft_type"], case["part_number"], case["airport"])
    case["step"] = nxt
    case["records"].append({"step": nxt, "found": found, "message": message, "detail": detail})


def _auto_sweep_silent_steps():
    case = st.session_state.case
    while not case["finished"] and case["step"] < TOTAL_STEPS:
        rec = case["records"][-1]
        if rec["found"]:
            break
        step = case["step"]
        if step <= 3:
            proceed_step(); continue
        if step == 4 and not rec["detail"].get("queried_airlines"):
            proceed_step(); continue
        break


def resolve_here():
    if st.session_state.case and not st.session_state.case["finished"]:
        st.session_state.case["finished"] = True


def _build_agent_report(case):
    checked = len(case["records"])
    action_section = build_rule_based_briefing(case).split("### 💡 추천 행동", 1)[-1].strip()
    return f"🔎 {case['registration']}({case['aircraft_type']}) · {case['airport']} — {checked}단계까지 확인했습니다.\n\n{action_section}"


# ----------------------------------------------------------------------------
# 8. 자연어 파싱 — 기번 / 파트넘버 / 공항
# ----------------------------------------------------------------------------

def parse_aog_message(text):
    upper = text.upper()
    compact = re.sub(r"\s+", "", upper)

    registration = next((r for r in REGISTRATION_CATALOG if r in compact), None)
    if not registration:
        m = re.search(r"\bHL[0-9A-Z]{3,4}\b", upper)
        if m:
            registration = m.group(0)
    if not registration:
        registration = next((t for t in AIRCRAFT_TYPES if norm(t).replace(" ", "") in compact), None)

    airport = None
    for ap, (_, _, name) in AIRPORT_COORDS.items():
        if re.search(rf"(?<![A-Z0-9]){ap}(?![A-Z0-9])", upper) or name in text:
            airport = ap
            break

    part_number = next((pn for pn in PART_NUMBER_CATALOG if pn in compact), None)
    if not part_number:
        m = re.search(r"\b[A-Z0-9]{2,}(?:-[A-Z0-9]+){1,}\b", upper)
        if m:
            part_number = m.group(0)

    mechanic = None
    m = re.search(r"(01[0-9]-?\d{3,4}-?\d{4})", text)
    if m:
        mechanic = m.group(1)

    return {"registration": registration, "part_number": part_number, "airport": airport, "mechanic_contact": mechanic}


# ----------------------------------------------------------------------------
# 9. 대시보드 화면
# ----------------------------------------------------------------------------

def render_case_intake():
    if "chat_prefill" in st.session_state:
        prefill = st.session_state.pop("chat_prefill")
        field_to_key = {"registration": "intake_reg", "part_number": "intake_pn",
                        "airport": "intake_airport", "mechanic_contact": "intake_mechanic"}
        for field, key in field_to_key.items():
            if prefill.get(field):
                st.session_state[key] = prefill[field]

    with st.expander("⌨️ 직접 입력 (채팅으로 일부만 인식됐거나 수동 입력 시)", expanded=not st.session_state.case):
        with st.form("case_form", clear_on_submit=False):
            c1, c2, c3 = st.columns(3)
            registration = c1.text_input("기번 (Registration)", placeholder="예: HL7702", key="intake_reg")
            part_number = c2.text_input("자재 파트넘버 (Part Number)", placeholder="예: OXY-GEN-A330-15", key="intake_pn")
            airport = c3.text_input("AOG 발생 공항 (Station)", placeholder="예: ICN", key="intake_airport")
            mechanic = st.text_input("현장 정비사 연락처 (선택)", placeholder="예: 010-1234-5678", key="intake_mechanic")
            submitted = st.form_submit_button("🚨 AOG 프로세스 시작", type="primary", use_container_width=True)

        if submitted:
            if not registration.strip() or not part_number.strip() or not airport.strip():
                st.warning("기번 / 자재 파트넘버 / 공항을 모두 입력해주세요.")
            else:
                ok, err = try_start_case(registration, part_number, airport, mechanic)
                if not ok:
                    st.error(err)
                    st.session_state.chat_log.append({"role": "assistant", "content": err})
                else:
                    c = st.session_state.case
                    st.session_state.chat_log.append({"role": "assistant",
                        "content": f"시작합니다 — 기번 **{c['registration']}**({c['aircraft_type']}) · 자재 **{c['part_number']}** · 공항 **{c['airport']}**."})
                    st.session_state.chat_log.append({"role": "assistant", "content": _build_agent_report(c)})
                    st.rerun()


def render_ai_panel():
    st.markdown("## 🤖 AI 상황 요약 & 추천 행동")
    case = st.session_state.case
    if not case:
        st.info("채팅 또는 직접 입력으로 케이스를 시작하면 이곳에 추천 행동이 즉시 표시됩니다.")
        return
    with st.container(border=True):
        st.markdown(build_rule_based_briefing(case))


def render_process_panel():
    st.markdown("## 📋 프로세스 진행")
    case = st.session_state.case
    if not case:
        st.info("케이스가 시작되지 않았습니다.")
        return

    last = case["records"][-1]
    with st.container(border=True):
        st.markdown(f"#### 진행 단계 {case['step']}/{TOTAL_STEPS} — {STEP_NAMES[case['step']]}")
        (st.success if last["found"] else st.warning)(last["message"])

        is_last = case["step"] >= TOTAL_STEPS
        bc1, bc2 = st.columns(2)
        if not case["finished"] and not is_last:
            if bc1.button("➡️ 아니요, 다음 대안 찾기", use_container_width=True, key="proceed_btn"):
                proceed_step(); _auto_sweep_silent_steps()
                st.session_state.chat_log.append({"role": "assistant", "content": _build_agent_report(st.session_state.case)})
                st.rerun()
        if not case["finished"]:
            if case["step"] in (4, 5):
                label = "✅ 대여 확보됨 (프로세스 종료)"
            elif is_last:
                label = "✅ Hand-carry 조치 확정 (프로세스 종료)"
            else:
                label = "✅ 네, 진행해주세요"
            if bc2.button(label, use_container_width=True, key="resolve_btn", type="primary"):
                resolve_here(); st.rerun()

    render_action_panel(case)

    with st.expander("📜 처리 이력 (Audit Log)", expanded=False):
        for rec in case["records"]:
            st.markdown(f"**{'✅' if rec['found'] else '❌'} {STEP_NAMES[rec['step']]}**\n\n{rec['message']}")
            st.divider()
        for entry in case["action_log"]:
            st.markdown(f"🗂️ {entry}")


def render_bulk_email_action(case, airlines, key_prefix):
    recipients = [a["email"] for a in airlines if a.get("email")]
    subject = DEFAULT_EMAIL_SUBJECT.format(
        aircraft_type=case["aircraft_type"], part_number=case["part_number"], airport=case["airport"])
    body_key = f"emailbody_{key_prefix}_{case['part_number']}_{case['aircraft_type']}_{case['airport']}"
    if body_key not in st.session_state:
        st.session_state[body_key] = DEFAULT_EMAIL_BODY_TEMPLATE.format(
            airport=case["airport"], aircraft_type=case["aircraft_type"], part_number=case["part_number"])

    st.write(f"문의 대상({len(airlines)}곳): {', '.join(a['airline'] for a in airlines) or '없음'}")
    if recipients:
        mailto = ("mailto:" + ",".join(recipients) + "?subject=" + urllib.parse.quote(subject)
                  + "&body=" + urllib.parse.quote(st.session_state[body_key]))
        st.markdown(f'<a href="{mailto}" target="_blank">✉️ 문의 대상 전체({len(recipients)}곳)에 지금 바로 메일 작성 (영문)</a>', unsafe_allow_html=True)
    else:
        st.warning("등록된 이메일이 없습니다. '데이터 관리'에서 항공사 이메일을 추가하세요.")

    with st.expander("메일 내용 확인/수정 (영문 템플릿으로 바로 발송 가능)"):
        st.text_area("Body", key=body_key, height=160, label_visibility="collapsed")
        if HAS_TRANSFORMERS:
            if st.button("✨ AI로 다듬기 (영문·선택)", key=f"ai_polish_{key_prefix}"):
                with st.spinner("AI가 영문 문구를 다듬는 중..."):
                    polished = generate_llm_email_body(case)
                if polished:
                    st.session_state[body_key] = polished; st.rerun()
                else:
                    st.warning("AI 응답을 못 받았습니다. 기본 영문 문구로 바로 발송하세요.")
        else:
            st.caption("ℹ️ transformers 미설치 — 기본 영문 템플릿으로 바로 발송 가능합니다.")

    if st.button("✉️ 긴급 메일 발송 완료로 기록", key=f"log_email_{key_prefix}"):
        case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] {len(recipients)}개 항공사에 긴급 메일(영문) 발송 완료")
        st.rerun()


def render_handcarry_standby(case):
    st.markdown("##### 🛡️ 만약을 대비한 6단계(Hand-carry) 사전 준비")
    st.caption("통관은 시간이 걸리므로, 이 단계 결과를 기다리지 말고 지금 통관팀에 여유 자재 불출을 병행 요청하는 것을 권장합니다.")
    customs = st.session_state.db["customs_team"]
    colA, colB = st.columns(2)
    colA.markdown(f'<a href="tel:{tel_href(customs["contact"])}">📞 통관팀 {customs["contact"]}</a>', unsafe_allow_html=True)
    colB.markdown(f'<a href="mailto:{customs["email"]}">✉️ {customs["email"]}</a>', unsafe_allow_html=True)
    flights = fetch_flight_schedule(case["airport"])[:2]
    if flights:
        preview = ", ".join(f"{f['flight_no']}(약 {f['hours_from_now']:.1f}h 후 도착)" for f in flights)
        st.caption(f"참고 — {case['airport']}행 가장 빠른 예정편: {preview}")
    else:
        st.caption(f"참고 — 현재 {case['airport']}행 예정편이 조회되지 않습니다. '실시간 운항 현황'에서 경유편을 확인하세요.")
    if st.button("☎️ 통관팀에 여유 자재 불출 사전 요청 완료로 기록", key=f"precall_{case['step']}"):
        case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] (사전 대비) 통관팀에 여유 자재 불출 사전 요청 완료")
        st.rerun()


def render_action_panel(case):
    st.markdown("#### 🎯 다음 조치")
    last = case["records"][-1]
    step, found, detail = last["step"], last["found"], last["detail"]

    if step == 1 and found:
        st.write(f"🧰 {case['aircraft_type']} {detail.get('kit_name', '표준 FAK')} · {detail.get('part_name', case['part_number'])} · 현장 {case['airport']} · 수량 {detail['qty']}개")
        if case["mechanic_contact"]:
            st.markdown(f'<a href="tel:{tel_href(case["mechanic_contact"])}">📞 현장 정비사({case["mechanic_contact"]})에게 전화</a>', unsafe_allow_html=True)
        else:
            st.caption("정비사 연락처가 없습니다. 위 정보로 직접 전달하세요.")
        if st.button("☎️ 정비사 전달 완료로 기록", key="log_step1"):
            case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] 현장 정비사에게 FAK 키트 자재 안내 완료"); st.rerun()

    elif step == 2 and found:
        dept = find_one(st.session_state.db["allocation_dept_contacts"], airport=case["airport"])
        st.write(f"📦 {detail.get('warehouse_name', '창고')} ({case['airport']}) · {detail.get('part_name', case['part_number'])} · 수량 {detail['qty']}개")
        if dept:
            st.write(f"담당: **{dept['department']}**")
            c1, c2 = st.columns(2)
            c1.markdown(f'<a href="tel:{tel_href(dept["contact"])}">📞 {dept["contact"]}</a>', unsafe_allow_html=True)
            c2.markdown(f'<a href="mailto:{dept["email"]}">✉️ {dept["email"]}</a>', unsafe_allow_html=True)
        else:
            st.warning(f"{case['airport']} Allocation 부서 연락처가 없습니다. '데이터 관리'에서 추가하세요.")
        if st.button("☎️ Allocation 부서 연락 완료로 기록", key="log_step2"):
            case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] {case['airport']} Allocation 부서 연락 완료"); st.rerun()

    elif step == 3 and found:
        st.write(f"🤝 {detail.get('part_name', case['part_number'])} · {detail['location']} · 수량 {detail['qty']}개 · 커버리지 {detail.get('coverage_airports', '전 공항')}")
        c1, c2 = st.columns(2)
        c1.markdown(f'<a href="tel:{tel_href(detail["contact"])}">📞 {detail["contact"]}</a>', unsafe_allow_html=True)
        if detail.get("email"):
            c2.markdown(f'<a href="mailto:{detail["email"]}">✉️ {detail["email"]}</a>', unsafe_allow_html=True)
        if st.button("☎️ Pooling 파트너사 연락 완료로 기록", key="log_step3"):
            case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] {detail['partner']}에 대여 연락 완료"); st.rerun()

    elif step in (4, 5):
        if step == 4:
            st.info("발생 공항 기반 항공사 중 우리 기종 운영사에 우선 문의합니다(가까워 빠른 지원 기대). 타사 실재고는 문의로 확인합니다.")
        else:
            st.info("기종을 운영하는 모든 타 항공사에 문의합니다. 타사 실재고는 기밀이라 개별 문의로만 확인됩니다.")
        render_bulk_email_action(case, detail.get("queried_airlines", []), f"step{step}")
        st.divider()
        render_handcarry_standby(case)

    elif step == 6:
        customs = detail["customs"]
        st.write(f"통관팀: **{customs['name']}** · {customs['contact']} · {customs['email']}")
        flights = detail["flights"]
        if flights:
            opts = [f"{f['flight_no']} — 약 {f['hours_from_now']:.1f}시간 후 도착 "
                    f"[{(datetime.now() + timedelta(hours=f['hours_from_now'])).strftime('%m/%d %H:%M')}]" for f in flights]
            chosen = st.radio("Hand-carry 편명 선택 (가장 빠른 편 상단)", opts, key="flight_choice")
            if st.button("✅ 이 편으로 Hand-carry 확정", key="log_step6"):
                case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] 통관팀 연계·Hand-carry 편명 확정: {chosen}"); st.rerun()
        else:
            st.warning(f"{case['airport']}행 예정편이 조회되지 않습니다. '실시간 운항 현황'에서 경유편을 확인하세요.")


_AOG_CSS = """
<style>
:root { --aog-navy:#0b2e4f; --aog-blue:#2a78d6; --aog-line:#e1e6ee; --aog-muted:#6b7684; }
.stApp { background:#f4f6f9; }
.block-container { padding-top:2.2rem; }
h1,h2,h3,h4 { color:var(--aog-navy); }
.aog-hero { background:linear-gradient(120deg,#0b2e4f 0%,#164b7e 100%); color:#fff;
  padding:18px 24px; border-radius:16px; margin-bottom:16px; box-shadow:0 6px 18px rgba(11,46,79,.18); }
.aog-hero .t { font-size:1.35rem; font-weight:800; letter-spacing:-.01em; }
.aog-hero .s { margin-top:4px; opacity:.85; font-size:.82rem; line-height:1.5; }
.aog-card { background:#fff; border:1px solid var(--aog-line); border-radius:14px;
  padding:14px 18px; margin-bottom:8px; box-shadow:0 1px 3px rgba(11,46,79,.05); }
.aog-kv { display:flex; flex-wrap:wrap; gap:22px; }
.aog-kv .k { font-size:.68rem; color:var(--aog-muted); text-transform:uppercase; letter-spacing:.05em; }
.aog-kv .v { font-size:1.06rem; font-weight:800; color:var(--aog-navy); margin-top:1px; }
.aog-pill { display:inline-block; padding:3px 12px; border-radius:999px; font-size:.74rem; font-weight:800; }
.aog-pill.ok { background:#e2f5e2; color:#0a7a0a; }
.aog-pill.no { background:#eef1f5; color:#7a8492; }
.aog-pill.ask { background:#fff3d6; color:#9a6b00; }
.aog-stepper { display:flex; margin:2px 0 16px; }
.aog-stepper .st { flex:1; text-align:center; position:relative; }
.aog-stepper .st:not(:first-child)::before { content:""; position:absolute; left:-50%; top:17px;
  width:100%; height:3px; background:var(--aog-line); z-index:0; }
.aog-stepper .st.on:not(:first-child)::before { background:var(--aog-blue); }
.aog-stepper .dot { width:36px; height:36px; border-radius:50%; margin:0 auto; display:flex;
  align-items:center; justify-content:center; font-weight:800; font-size:.92rem; position:relative;
  z-index:1; border:3px solid var(--aog-line); background:#fff; color:#9aa5b1; }
.aog-stepper .st.cur .dot { border-color:var(--aog-blue); background:var(--aog-blue); color:#fff;
  box-shadow:0 0 0 4px rgba(42,120,214,.18); }
.aog-stepper .st.found .dot { border-color:#0ca30c; background:#0ca30c; color:#fff; }
.aog-stepper .lbl { font-size:.66rem; color:var(--aog-muted); margin-top:6px; line-height:1.2; }
.aog-stepper .st.cur .lbl { color:var(--aog-navy); font-weight:800; }
[data-testid="stMetric"] { background:#fff; border:1px solid var(--aog-line); border-radius:12px;
  padding:12px 14px; box-shadow:0 1px 3px rgba(11,46,79,.05); }
[data-testid="stMetricLabel"] p { color:var(--aog-muted); font-weight:700; }
</style>
"""

_STEP_SHORT = {1: "FAK", 2: "Allocation", 3: "Pooling", 4: "Main Station", 5: "동일기종", 6: "Hand-carry"}


def render_stepper(case):
    recs = {r["step"]: r for r in case["records"]}
    cur = case["step"]
    cells = []
    for s in range(1, TOTAL_STEPS + 1):
        rec = recs.get(s)
        classes = ["st"]
        if s <= cur:
            classes.append("on")
        if rec is not None and rec["found"]:
            classes.append("found"); icon = "✓"
        elif s == cur and not case["finished"]:
            classes.append("cur"); icon = str(s)
        elif rec is not None:
            icon = "✕"
        else:
            icon = str(s)
        cells.append(f'<div class="{" ".join(classes)}"><div class="dot">{icon}</div>'
                     f'<div class="lbl">{s}. {_STEP_SHORT[s]}</div></div>')
    st.markdown(f'<div class="aog-stepper">{"".join(cells)}</div>', unsafe_allow_html=True)


def render_case_header(case):
    last = case["records"][-1]
    if case["finished"] and last["found"]:
        pill = '<span class="aog-pill ok">✅ 자재 확보</span>'
    elif case["finished"]:
        pill = '<span class="aog-pill no">⏹ 종료(외부 확보/대체)</span>'
    elif last["found"]:
        pill = '<span class="aog-pill ok">✅ 확보 가능 · 승인 대기</span>'
    elif case["step"] in (4, 5):
        pill = '<span class="aog-pill ask">✉️ 타사 문의 필요</span>'
    else:
        pill = '<span class="aog-pill ask">🔎 조회 진행 중</span>'
    st.markdown(
        f'<div class="aog-card"><div class="aog-kv">'
        f'<div><div class="k">기번 → 기종</div><div class="v">{case["registration"]} · {case["aircraft_type"]}</div></div>'
        f'<div><div class="k">자재 파트넘버</div><div class="v">{case["part_number"]}</div></div>'
        f'<div><div class="k">발생 공항</div><div class="v">{case["airport"]}</div></div>'
        f'<div><div class="k">현재 상태</div><div class="v">{pill}</div></div>'
        f'</div></div>', unsafe_allow_html=True)
    render_stepper(case)


def render_history_panel(case):
    db = st.session_state.db
    pn = case["part_number"]
    sh = sorted([r for r in db.get("sourcing_history", []) if norm(r.get("part_number")) == norm(pn)],
                key=lambda r: r.get("date", ""), reverse=True)
    dh = sorted([r for r in db.get("defect_history", []) if norm(r.get("part_number")) == norm(pn)],
                key=lambda r: r.get("date", ""), reverse=True)
    st.markdown(f"### 📖 이 자재 참고자료 — `{pn}`  ·  과거에 어떻게 처리했는지")
    c1, c2 = st.columns(2)
    with c1:
        st.markdown("**🔁 과거 수배(빌린) 이력** — 어디서·어떤 방식으로 확보했는지 · 성공/실패")
        if sh:
            ok = sum(1 for r in sh if str(r.get("result", "")).startswith("성공"))
            top_src = Counter(r.get("source", "") for r in sh).most_common(1)[0][0]
            st.caption(f"📌 과거 {len(sh)}건 · 성공 {ok}건 · 최다 수배처: **{top_src}**")
            df = pd.DataFrame(sh)[["date", "airport", "resolved_step", "source", "method", "result", "lead_time_hours"]]
            df.columns = ["일자", "공항", "확보 단계", "수배처", "방식", "결과", "소요(h)"]
            st.dataframe(df, use_container_width=True, hide_index=True)
        else:
            st.caption("이 자재의 과거 수배 이력이 없습니다.")
    with c2:
        st.markdown("**🛠️ 과거 결함 조치 이력** — 교환/디퍼/리셋 · 단일 vs 패키지 · 전용 공구")
        if dh:
            top_res = ", ".join(f"{k}({v})" for k, v in Counter(r.get("resolution", "") for r in dh).most_common())
            tools = [r.get("tools_required", "") for r in dh if r.get("tools_required") and r.get("tools_required") != "표준 공구"]
            cap = f"📌 과거 {len(dh)}건 · 조치 분포: {top_res}"
            if tools:
                cap += f" · 전용공구 예: **{tools[0]}**"
            st.caption(cap)
            df = pd.DataFrame(dh)[["date", "registration", "defect", "resolution", "parts_scope", "tools_required", "result", "downtime_hours"]]
            df.columns = ["일자", "기번", "결함", "조치유형", "수배범위", "전용공구", "결과", "다운타임(h)"]
            st.dataframe(df, use_container_width=True, hide_index=True)
        else:
            st.caption("이 자재의 과거 결함 조치 이력이 없습니다.")


def render_dashboard_page():
    st.markdown(
        '<div class="aog-hero"><div class="t">🚨 AOG 자재 수급 어시스턴트</div>'
        '<div class="s">기번·자재 파트넘버·공항을 말하면 저장된 데이터를 근거로 1단계부터 순차 조회합니다. '
        'FAK→Allocation→Pooling은 재고가 없으면 자동으로 다음 단계까지 확인하고, '
        '타사 문의(4·5)와 Hand-carry(6)에서만 멈춰 승인을 기다립니다.</div></div>',
        unsafe_allow_html=True)

    for msg in st.session_state.chat_log:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

    render_case_intake()

    if st.session_state.case:
        render_case_header(st.session_state.case)
        left, right = st.columns(2)
        with left:
            render_ai_panel()
        with right:
            render_process_panel()
        st.divider()
        render_history_panel(st.session_state.case)
        st.divider()
        if st.button("🔄 새 AOG 건 시작 (초기화)"):
            st.session_state.case = None
            st.session_state.chat_log = []
            st.rerun()

    user_msg = st.chat_input("예: HL7702 ICN에서 AOG, 부품 OXY-GEN-A330-15 필요")
    if user_msg:
        st.session_state.chat_log.append({"role": "user", "content": user_msg})
        parsed = parse_aog_message(user_msg)
        label = {"registration": "기번", "part_number": "자재 파트넘버", "airport": "공항"}
        missing = [k for k in ("registration", "part_number", "airport") if not parsed[k]]
        if missing:
            recog = ", ".join(f"{label[k]} **{v}**" for k, v in parsed.items() if v and k in label)
            reply = ((f"인식: {recog}\n\n" if recog else "") +
                     f"**{', '.join(label[k] for k in missing)}**를 못 찾았어요. 아래 '직접 입력'에서 마저 채워주세요.")
            st.session_state.chat_log.append({"role": "assistant", "content": reply})
            st.session_state.chat_prefill = parsed
        else:
            ok, err = try_start_case(parsed["registration"], parsed["part_number"], parsed["airport"], parsed.get("mechanic_contact") or "")
            if not ok:
                st.session_state.chat_log.append({"role": "assistant", "content": err})
                st.session_state.chat_prefill = parsed
            else:
                c = st.session_state.case
                st.session_state.chat_log.append({"role": "assistant",
                    "content": f"인식했습니다 — 기번 **{c['registration']}**({c['aircraft_type']}) · 자재 **{c['part_number']}** · 공항 **{c['airport']}**. FAK부터 순차 조회할게요."})
                st.session_state.chat_log.append({"role": "assistant", "content": _build_agent_report(c)})
        st.rerun()


# ----------------------------------------------------------------------------
# 10. 데이터 관리 화면
# ----------------------------------------------------------------------------

def df_editor_save(key, columns, int_cols=()):
    db = st.session_state.db
    df = pd.DataFrame(db[key])
    if df.empty:
        df = pd.DataFrame(columns=columns)
    edited = st.data_editor(df, num_rows="dynamic", use_container_width=True, key=f"editor_{key}")
    if st.button("💾 저장", key=f"save_{key}"):
        cleaned = edited.fillna("")
        for c in int_cols:
            if c in cleaned.columns:
                cleaned[c] = pd.to_numeric(cleaned[c], errors="coerce").fillna(0).astype(int)
        db[key] = cleaned.to_dict("records")
        save_db(db)
        st.success("저장되었습니다.")


def render_bundle_editor(db_key, group_field, group_label, name_field, content_cols, int_cols=()):
    """묶음(패키지/창고) 편집기: 그룹(기종/공항)을 고르면 그 안의 contents를 표로 편집."""
    db = st.session_state.db
    bundles = db.setdefault(db_key, [])
    if bundles:
        def fmt(i):
            b = bundles[i]
            nm = f" · {b.get(name_field, '')}" if name_field else ""
            return f"{b.get(group_field, '?')}{nm} ({len(b.get('contents', []))}종)"
        idx = st.selectbox(f"{group_label} 선택", list(range(len(bundles))), format_func=fmt, key=f"sel_{db_key}")
        bundle = bundles[idx]
        if name_field:
            bundle[name_field] = st.text_input("묶음 이름", value=bundle.get(name_field, ""), key=f"name_{db_key}_{idx}")
        df = pd.DataFrame(bundle.get("contents", []))
        if df.empty:
            df = pd.DataFrame(columns=content_cols)
        edited = st.data_editor(df, num_rows="dynamic", use_container_width=True, key=f"ed_{db_key}_{idx}")
        if st.button("💾 이 묶음 저장", key=f"save_{db_key}_{idx}"):
            cleaned = edited.fillna("")
            for c in int_cols:
                if c in cleaned.columns:
                    cleaned[c] = pd.to_numeric(cleaned[c], errors="coerce").fillna(0).astype(int)
            bundle["contents"] = cleaned.to_dict("records")
            save_db(db); st.success("저장되었습니다.")
    else:
        st.info("등록된 묶음이 없습니다.")
    with st.expander(f"➕ 새 {group_label} 추가"):
        newg = st.text_input(f"{group_field} 값", key=f"newg_{db_key}")
        newn = st.text_input("묶음 이름", key=f"newn_{db_key}") if name_field else None
        if st.button("추가", key=f"add_{db_key}"):
            if newg.strip():
                nb = {group_field: newg.strip(), "contents": []}
                if name_field:
                    nb[name_field] = (newn or "").strip()
                bundles.append(nb); save_db(db); st.rerun()


def render_admin_page():
    st.markdown(
        '<div class="aog-hero"><div class="t">⚙️ 데이터 관리</div>'
        '<div class="s">여기 저장된 데이터가 프로세스 판단의 유일한 근거입니다. 저장 즉시 대시보드 조회에 반영됩니다.</div></div>',
        unsafe_allow_html=True)
    if os.path.exists(DB_PATH):
        st.caption(f"마지막 저장: {datetime.fromtimestamp(os.path.getmtime(DB_PATH)).strftime('%Y-%m-%d %H:%M')}")

    tabs = st.tabs([
        "🛩️ 기번 등록부", "🧰 FAK 키트(기종별 패키지)", "📦 Allocation(스테이션 창고)", "🤝 Pooling",
        "🛫 Main Station 항공사", "✈️ 기종별 운영 항공사", "📇 Allocation 부서·통관팀",
        "🔁 수배 이력", "🛠️ 결함 조치 이력",
    ])
    with tabs[0]:
        st.caption("기번 → 기종 매핑. 입력은 기번으로, 타사 문의는 여기서 변환된 기종으로 나갑니다.")
        df_editor_save("aircraft_registry", ["registration", "aircraft_type"])
    with tabs[1]:
        st.caption("같은 기종은 모두 '동일한 키트 하나(여러 자재 패키지)'를 기체와 함께 탑재합니다. "
                   "기종을 고르고 그 키트 안의 자재 묶음을 편집하세요. 발생 공항과 무관하게 사용 가능으로 판단합니다.")
        render_bundle_editor("fak_kits", "aircraft_type", "FAK 키트(기종)", "kit_name",
                             ["part_number", "part_name", "qty"], int_cols=["qty"])
    with tabs[2]:
        st.caption("한 스테이션(공항) 창고 안에 여러 자재가 들어있습니다. 공항을 고르고 그 창고 안의 자재 묶음을 "
                   "편집하세요. 발생 공항 창고에 있어야만 사용 가능하며, 없으면 Pooling으로 넘어갑니다.")
        render_bundle_editor("station_warehouses", "airport", "스테이션 창고(공항)", "warehouse_name",
                             ["aircraft_type", "part_number", "part_name", "qty"], int_cols=["qty"])
    with tabs[3]:
        st.markdown("**파트너사 목록**")
        st.caption("`coverage_airports`: 지원 가능한 공항 코드 콤마 구분(예: `SIN,HKG,BKK`). 비우면 전 공항 지원.")
        df_editor_save("pooling_partners", ["partner", "contact", "email", "coverage_airports"])
        st.markdown("**파트너사 보유 재고**")
        df_editor_save("pooling_stock", ["partner", "aircraft_type", "part_number", "part_name", "qty", "location"], int_cols=["qty"])
    with tabs[4]:
        st.caption("공항별 Main Station 항공사. 4단계는 여기 항공사 ∩ 우리 기종 운영사로 문의 대상을 좁힙니다.")
        df_editor_save("main_station_airlines", ["airport", "airline", "contact", "email"])
    with tabs[5]:
        st.caption("기종별 운영 항공사(5단계 문의 대상 + 4단계 교집합 판단 근거). 타사 실재고는 기밀이라 관리하지 않습니다.")
        df_editor_save("fleet_operators", ["aircraft_type", "airline", "contact", "email"])
    with tabs[6]:
        st.markdown("**공항별 Allocation 부서 연락처**")
        df_editor_save("allocation_dept_contacts", ["airport", "department", "contact", "email"])
        st.markdown("**통관팀 연락처**")
        db = st.session_state.db
        c1, c2, c3 = st.columns(3)
        name = c1.text_input("담당팀명", value=db["customs_team"]["name"], key="customs_name")
        contact = c2.text_input("연락처", value=db["customs_team"]["contact"], key="customs_contact")
        email = c3.text_input("이메일", value=db["customs_team"]["email"], key="customs_email")
        if st.button("💾 저장", key="save_customs"):
            db["customs_team"] = {"name": name, "contact": contact, "email": email}
            save_db(db); st.success("저장되었습니다.")
    with tabs[7]:
        st.caption("자재별 과거 수배(빌린) 이력 — 어디서·어떤 방식(대여/구매/Hand-carry/자체재고)으로 확보했고 성공/실패했는지. "
                   "대시보드에서 해당 자재 케이스 진행 시 자동으로 참고자료로 표시됩니다.")
        df_editor_save("sourcing_history",
                       ["date", "registration", "aircraft_type", "part_number", "airport", "resolved_step", "source", "method", "result", "lead_time_hours"],
                       int_cols=["lead_time_hours"])
    with tabs[8]:
        st.caption("자재별 과거 결함 조치 이력 — 조치유형(교환/디퍼/리셋·재시동/SW 리로드), 수배범위(단일 부품 vs 패키지 어셈블리), "
                   "전용 공구까지. 대시보드에서 해당 자재 케이스 진행 시 자동으로 참고자료로 표시됩니다.")
        df_editor_save("defect_history",
                       ["date", "registration", "aircraft_type", "part_number", "defect", "resolution", "parts_scope", "tools_required", "result", "downtime_hours"],
                       int_cols=["downtime_hours"])


# ----------------------------------------------------------------------------
# 11. 실시간 운항 현황 — 허브/주력 노선 시인성
# ----------------------------------------------------------------------------

def build_flight_map(flights, parking, movements):
    fig = go.Figure()
    route_freq = Counter()
    for f in flights:
        route_freq[tuple(sorted([f["origin"], f["destination_airport"]]))] += 1
    max_rf = max(route_freq.values()) if route_freq else 1
    for (a, b), c in route_freq.items():
        if a not in AIRPORT_COORDS or b not in AIRPORT_COORDS:
            continue
        la, lo, _ = AIRPORT_COORDS[a]; lb, lob, _ = AIRPORT_COORDS[b]
        fig.add_trace(go.Scattergeo(lat=[la, lb], lon=[lo, lob], mode="lines",
            line=dict(width=1 + 5 * c / max_rf, color="#2a78d6"), opacity=0.4, hoverinfo="skip", showlegend=False))

    max_mv = max(movements.values()) if movements else 1
    lats, lons, texts, sizes, colors = [], [], [], [], []
    for code, (lat, lon, name) in AIRPORT_COORDS.items():
        mv = movements.get(code, 0)
        info = parking.get(code, {"total_stands": 0, "occupied": 0})
        rate = info["occupied"] / info["total_stands"] if info["total_stands"] else 0
        lats.append(lat); lons.append(lon)
        texts.append(f"{name}({code})<br>운항량 {mv}편 · 정박 {info['occupied']}/{info['total_stands']} ({rate:.0%})")
        sizes.append(10 + 26 * (mv / max_mv))
        colors.append(rate)
    fig.add_trace(go.Scattergeo(lat=lats, lon=lons, text=texts, mode="markers", hoverinfo="text", name="공항",
        marker=dict(size=sizes, color=colors, colorscale=[[0, "#0ca30c"], [0.7, "#fab219"], [1, "#d03b3b"]],
                    cmin=0, cmax=1, showscale=True, colorbar=dict(title="정박<br>혼잡도", tickformat=".0%"),
                    line=dict(width=1, color="#ffffff"))))

    for f in flights:
        if f["status"] != "비행 중" or f["origin"] not in AIRPORT_COORDS or f["destination_airport"] not in AIRPORT_COORDS:
            continue
        olat, olon, _ = AIRPORT_COORDS[f["origin"]]; dlat, dlon, _ = AIRPORT_COORDS[f["destination_airport"]]
        plat = olat + (dlat - olat) * f["progress"]; plon = olon + (dlon - olon) * f["progress"]
        fig.add_trace(go.Scattergeo(lat=[plat], lon=[plon], mode="markers", showlegend=False, hoverinfo="text",
            text=f"{f['flight_no']} {f['origin']}→{f['destination_airport']} ({f['progress']:.0%})",
            marker=dict(size=9, symbol="triangle-up", color="#eb6834", line=dict(width=1, color="#ffffff"))))

    fig.update_geos(projection_type="natural earth", showland=True, landcolor="#f0efec",
                    showcountries=True, countrycolor="#c3c2b7", showocean=True, oceancolor="#fcfcfb")
    fig.update_layout(height=560, margin=dict(l=0, r=0, t=10, b=0),
                      font=dict(family="system-ui, -apple-system, 'Segoe UI', sans-serif"))
    return fig


_STATUS_ICON = {"비행 중": "🛫 비행 중", "예정": "🕐 예정", "도착": "✅ 도착"}
_STATUS_ORDER = {"비행 중": 0, "예정": 1, "도착": 2}


def render_map_page():
    st.markdown(
        '<div class="aog-hero"><div class="t">🗺️ 실시간 운항 & 허브 현황</div>'
        '<div class="s">어느 공항이 허브이고 어떤 노선이 주력인지 한눈에 보여줍니다. 더미 데이터이며, '
        '실 API 연동 시 코드 변경 없이 그대로 반영됩니다.</div></div>', unsafe_allow_html=True)

    if st.button("🔄 새로고침"):
        _fetch_raw_flight_feed.clear(); fetch_airport_parking_status.clear(); st.rerun()

    flights = fetch_live_flights()
    parking = fetch_airport_parking_status()

    movements = Counter()
    route_freq = Counter()
    for f in flights:
        movements[f["origin"]] += 1
        movements[f["destination_airport"]] += 1
        route_freq[tuple(sorted([f["origin"], f["destination_airport"]]))] += 1

    in_flight = sum(1 for f in flights if f["status"] == "비행 중")
    top_hub = movements.most_common(1)[0] if movements else ("-", 0)
    top_route = route_freq.most_common(1)[0] if route_freq else (("-", "-"), 0)
    hub_name = AIRPORT_COORDS.get(top_hub[0], (0, 0, top_hub[0]))[2]
    k1, k2, k3, k4 = st.columns(4)
    k1.metric("전체 항공편", f"{len(flights)}편")
    k2.metric("🛫 운항 중", f"{in_flight}편")
    k3.metric("최다 허브", f"{hub_name}({top_hub[0]})", f"{top_hub[1]}편 취항")
    k4.metric("주력 노선", f"{top_route[0][0]}–{top_route[0][1]}", f"{top_route[1]}편")

    st.plotly_chart(build_flight_map(flights, parking, movements), use_container_width=True)

    col1, col2 = st.columns(2)
    with col1:
        st.markdown("#### 🏆 공항별 운항량 (허브 순위)")
        rows = [{"공항": f"{AIRPORT_COORDS.get(a, (0, 0, a))[2]}({a})", "운항량(편)": c,
                 "비중": c / sum(movements.values()) if movements else 0}
                for a, c in movements.most_common()]
        st.dataframe(pd.DataFrame(rows), use_container_width=True, hide_index=True,
                     column_config={"비중": st.column_config.ProgressColumn("비중", min_value=0, max_value=max([r["비중"] for r in rows] + [1]), format="%.0f%%")})
    with col2:
        st.markdown("#### 🔗 주력 노선 Top")
        rrows = [{"노선": f"{a}–{b}", "편수": c,
                  "비중": c / sum(route_freq.values()) if route_freq else 0}
                 for (a, b), c in route_freq.most_common(10)]
        st.dataframe(pd.DataFrame(rrows), use_container_width=True, hide_index=True,
                     column_config={"비중": st.column_config.ProgressColumn("비중", min_value=0, max_value=max([r["비중"] for r in rrows] + [1]), format="%.0f%%")})

    st.markdown("#### ✈️ 실시간 항공편")
    df = pd.DataFrame(flights)
    df["_o"] = df["status"].map(_STATUS_ORDER)
    df = df.sort_values(["_o", "hours_from_now"])
    df["상태"] = df["status"].map(_STATUS_ICON)
    df["출발"] = df["dep_time"].dt.strftime("%m/%d %H:%M")
    df["도착 예정"] = df["arr_time"].dt.strftime("%m/%d %H:%M")
    show = df[["flight_no", "origin", "destination_airport", "상태", "progress", "출발", "도착 예정"]]
    show.columns = ["편명", "출발", "도착", "상태", "진행률", "출발시각", "도착예정"]
    st.dataframe(show, use_container_width=True, hide_index=True,
                 column_config={"진행률": st.column_config.ProgressColumn("진행률", min_value=0, max_value=1, format="%.0f%%")})


# ----------------------------------------------------------------------------
# 12. 사이드바 내비게이션
# ----------------------------------------------------------------------------

st.markdown(_AOG_CSS, unsafe_allow_html=True)

with st.sidebar:
    st.markdown("## ✈️ AOG 어시스턴트")
    page = st.radio("화면 선택", ["🚨 AOG 대시보드", "🗺️ 실시간 운항 현황", "⚙️ 데이터 관리"], key="nav_page")
    st.divider()
    st.caption("⚡ 모든 판단은 '데이터 관리'의 데이터로만 이루어집니다. 타사 문의 메일은 영문이 기본입니다.")
    st.caption("Engine: Streamlit · JSON 영속화 · 항공편 API 연동 포인트 내장")

if page == "🚨 AOG 대시보드":
    render_dashboard_page()
elif page == "🗺️ 실시간 운항 현황":
    render_map_page()
else:
    render_admin_page()


## Cell 3 — 앱 실행 (Colab 자체 포트포워딩만 사용, 외부 터널 불필요)

npm/localtunnel 같은 제3자 서비스를 전혀 쓰지 않고, Google Colab이 공식 제공하는 커널 포트포워딩(`google.colab.output`)만 사용합니다. 이미 접속 중인 `colab.research.google.com` 도메인을 그대로 경유하므로 사내망 보안 정책에 걸릴 일이 없습니다.

In [ ]:
import subprocess, time

LOG_PATH = "/content/streamlit_log.txt"

# 기존에 떠 있던 프로세스가 있으면 정리
subprocess.run(["pkill", "-f", "streamlit run app.py"], stderr=subprocess.DEVNULL)

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.enableWebsocketCompression", "false",
    ],
    stdout=open(LOG_PATH, "w"),
    stderr=subprocess.STDOUT,
)

booted = False
for _ in range(30):
    time.sleep(1)
    log_text = open(LOG_PATH).read()
    if "You can now view your Streamlit app" in log_text or "Local URL" in log_text:
        booted = True
        break

if not booted:
    print("Streamlit이 기동되지 않았습니다. 아래 로그를 확인하세요.\n")
    print(log_text)
else:
    from google.colab import output
    print("Streamlit 정상 기동. 아래에서 새 창으로 대시보드를 엽니다.")
    output.serve_kernel_port_as_window(8501)

    # 새 창 팝업이 막혀 있다면 아래 줄의 주석을 풀어 노트북 내부에 바로 임베드할 수 있습니다.
    # output.serve_kernel_port_as_iframe(8501, height=900)
